
# Building GPT From Scratch: A Character-Level Decoder-Only Transformer

**Project type:** From-scratch implementation of a GPT-style (Generative Pre-trained Transformer) language model.

**Inspiration:** This project follows the structure of Andrej Karpathy's *"Let's build GPT: from scratch, in code, spelled out"* lecture, but every section below is re-derived, re-explained, and re-organized so that a reader with basic Python + linear algebra + a little calculus/statistics can follow it end-to-end and understand **why** each piece exists, not just copy it.

## What is a GPT and why build one ourselves?

ChatGPT and similar systems feel almost magical: you type a prompt, and coherent, sequential text streams out. Underneath, though, the mechanism is a **language model** — a system that predicts the *next token* given the tokens that came before it, over and over, one token at a time.

The neural network architecture that makes modern LLMs possible is the **Transformer**, introduced in the 2017 paper *"Attention Is All You Need"*. GPT (Generative Pre-trained Transformer) is essentially:

1. A **decoder-only Transformer** (we'll explain exactly what "decoder-only" means below),
2. **Pre-trained** on a huge amount of text to simply predict the next token,
3. Later **fine-tuned** (via supervised fine-tuning + RLHF) to become a helpful assistant like ChatGPT.

This notebook builds stage (1) and (2) — the pre-training of a small decoder-only transformer — completely from scratch in PyTorch. We will **not** implement the fine-tuning/RLHF stage that turns a raw language model into an assistant, because that requires human preference data and infrastructure outside the scope of this project (we explain what it involves at the very end, conceptually).

## Why build a *tiny* version instead of the real GPT-2/GPT-3?

Real GPT models (GPT-2: 124M-1.5B parameters, GPT-3: up to 175B parameters) are trained on hundreds of billions of tokens using thousands of GPUs. That's not reproducible on a laptop or a single Colab GPU. Instead, the pedagogical trick (borrowed from Karpathy's `nanoGPT`) is to:

- Use a **tiny dataset** (~1MB of Shakespeare's complete works, called **Tiny Shakespeare**),
- Use a **character-level tokenizer** instead of a subword tokenizer (so the vocabulary is only ~65 symbols instead of ~50,000),
- Train a **small transformer** (~10M parameters) that fits comfortably on a single GPU (or even a CPU, slowly).

The resulting model won't write real Shakespeare -- it will write text that *looks* like Shakespeare typographically (character names, dialogue structure, archaic words) without being coherent. But architecturally, it is **the same thing as GPT-2/GPT-3**, just scaled down. Everything you learn here -- tokenization, batching, self-attention, multi-head attention, residual connections, layer norm -- transfers directly to the real, production-scale models. At the end of this notebook we also show exactly how to reconfigure this code to match the real **GPT-2 124M** hyperparameters, as a bonus extension.

## Roadmap

| Section | What we build | Why we need it |
|---|---|---|
| 1 | Data loading & character tokenizer | Convert raw text into integers a neural net can consume |
| 2 | Train/val split & batching | Feed the model fixed-size chunks efficiently, in parallel |
| 3 | Bigram baseline model | Simplest possible "language model" as a sanity-check baseline |
| 4 | The "weighted average via matrix multiply" trick | The mathematical core that makes self-attention efficient |
| 5 | Single-head self-attention | Let tokens exchange information based on *content*, not just position |
| 6 | Multi-head attention | Let tokens communicate along several independent "channels" at once |
| 7 | Feed-forward network | Give each token time to "think" after gathering information |
| 8 | Transformer block (residuals + LayerNorm) | Let us stack many layers without the network becoming untrainable |
| 9 | Full GPT model | Assemble everything into one class |
| 10 | Training loop | Actually optimize the parameters |
| 11 | Text generation | Sample new Shakespeare-like text from the trained model |
| 12 | Scaling to GPT-2 (124M) config | Bonus: show exactly what changes to reach real GPT-2 scale |
| 13 | Loading real pretrained GPT-2 weights | Prove our architecture is correct by making it predict fluent English like real GPT-2 |
| 14 | Bonus: RoPE (rotary position embeddings) | A working, testable alternative to absolute position embeddings |
| 15 | Bonus: multilingual tokenization | Show the tokenizer/model handling more than just English |
| 16 | What's not included: fine-tuning / RLHF | An honest, concrete look at the gap between this model and an assistant like ChatGPT |



## 0. Setup

We use PyTorch as our only real dependency. We also set a manual seed everywhere for reproducibility -- since training involves random batch sampling and random weight initialization, fixing the seed means you should get (nearly) the same numbers we describe in the markdown below.


In [ ]:

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("WARNING: No GPU detected. Training will be slow -- consider reducing "
          "the hyperparameters below (n_embd, n_layer, n_head, block_size, batch_size).")


Using device: cuda



## 1. Dataset & Tokenization

### Why this dataset?

We use **Tiny Shakespeare**: a ~1MB text file that concatenates the complete works of Shakespeare into a single file. It's small enough to train on quickly, but has enough internal structure (character names, dialogue, meter, repeated vocabulary) that a model can visibly learn patterns within minutes, which makes it ideal for learning the *mechanics* of a transformer rather than fighting with a massive, slow data pipeline.

### Why character-level tokenization?

"Tokenizing" means turning raw text (a string) into a sequence of integers a neural network can process, using some fixed **vocabulary**. Production systems like GPT-2/3 use **byte-pair encoding (BPE)**, e.g. OpenAI's `tiktoken` library, which chunks text into ~50,000 possible subword pieces. This gives short sequences (efficient) but a large, non-trivial vocabulary/embedding table.

For learning purposes we instead tokenize at the **character level**: every unique character in the dataset becomes one vocabulary entry. This has a tiny vocabulary (~65 symbols: letters, punctuation, whitespace) and a dead-simple encode/decode scheme, at the cost of longer token sequences. This trade-off (vocab size vs. sequence length) is a real, general design decision in tokenizer design -- just simplified here for clarity.


In [ ]:

# Download Tiny Shakespeare (same dataset used in Karpathy's char-rnn / nanoGPT work)
import os
import urllib.request

data_path = "input.txt"
if not os.path.exists(data_path):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, data_path)

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")
print("---- First 300 characters ----")
print(text[:300])


Length of dataset in characters: 1115394
---- First 300 characters ----
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [ ]:

# Build the vocabulary: every unique character that appears in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Vocabulary: {chars}")

# Build encoder/decoder lookup tables
stoi = {ch: i for i, ch in enumerate(chars)}   # string-to-integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer-to-string

encode = lambda s: [stoi[c] for c in s]          # string -> list[int]
decode = lambda l: ''.join([itos[i] for i in l]) # list[int] -> string

print(encode("hi there"))
print(decode(encode("hi there")))


Vocabulary size: 65
Vocabulary: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [ ]:

# Encode the entire dataset and split into train/validation sets
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

n = int(0.9 * len(data))   # first 90% for training
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data)}, Val tokens: {len(val_data)}")


torch.Size([1115394]) torch.int64
Train tokens: 1003854, Val tokens: 111540



**Why a train/val split?** We hold out 10% of the data (never trained on) so we can measure whether the model is genuinely learning the *statistical patterns* of Shakespearean English, versus simply memorizing the exact training text. If validation loss stops improving while training loss keeps dropping, that's the classic signature of overfitting.



## 2. Chunking & Batching

We never feed the model the *entire* text at once -- that would be computationally prohibitive (the self-attention cost grows quadratically with sequence length, as we'll see). Instead we train on small, randomly sampled chunks.

- **`block_size`** (a.k.a. *context length*): the maximum number of previous characters the model is allowed to look at when predicting the next one.
- **`batch_size`**: how many independent chunks we process **in parallel** in a single forward/backward pass, purely for GPU efficiency (chunks in a batch never interact with each other).

A subtlety worth internalizing: a chunk of `block_size + 1` characters actually contains `block_size` *separate* training examples, since every prefix within the chunk gives us a `(context -> next character)` pair. Training on all of these -- from a context of length 1 up to length `block_size` -- is what teaches the transformer to make good predictions no matter how much context it currently has, which matters at generation time when we start from very little context.


In [ ]:

block_size = 8   # small value just to illustrate the concept; we scale this up later
x = train_data[:block_size]
y = train_data[1:block_size+1]   # y is x shifted by one position: the "next char" targets

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is: {target.item()}")


when input is [18] the target is: 47
when input is [18, 47] the target is: 56
when input is [18, 47, 56] the target is: 57
when input is [18, 47, 56, 57] the target is: 58
when input is [18, 47, 56, 57, 58] the target is: 1
when input is [18, 47, 56, 57, 58, 1] the target is: 15
when input is [18, 47, 56, 57, 58, 1, 15] the target is: 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target is: 58


In [ ]:

torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    """Sample a random batch of (input, target) chunks from train or val data."""
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))       # random starting offsets
    x = torch.stack([data[i:i+block_size] for i in ix])             # inputs
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])         # targets (shifted by 1)
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')



## 3. Baseline: A Bigram Language Model

Before jumping to a full transformer, we build the simplest possible neural language model as a baseline: a **bigram model**. It predicts the next character using *only the identity of the current character* -- no context, no communication between tokens at all.

**Why bother with something this weak?**
1. It gives us a working end-to-end pipeline (data -> model -> loss -> generation) before we add complexity.
2. It gives us a concrete number to beat: if our fancy transformer can't do better than the bigram model, something is wrong.
3. With `vocab_size = 65` uniformly-random logits, we can compute the *expected* initial loss analytically as `-log(1/65) ~= 4.17`. Comparing our actual initial loss to this tells us whether our loss function is wired up correctly.

Mechanically, the model is just a lookup table (`nn.Embedding`) of shape `(vocab_size, vocab_size)`: given a token index, it directly returns a row of raw scores ("logits") for what the *next* token might be.


In [ ]:

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C) where C = vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]                    # focus only on the last time step: (B, C)
            probs = F.softmax(logits, dim=-1)             # convert logits -> probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  # sample next token: (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)       # append to the running sequence
        return idx

m = BigramLanguageModel(vocab_size).to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(f"loss = {loss.item():.4f}  (expected ~{-torch.log(torch.tensor(1.0/vocab_size)):.4f} at random init)")

# Generate from an untrained model -- expect pure noise
idx0 = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with newline token
print(decode(m.generate(idx0, max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
loss = 5.0364  (expected ~4.1744 at random init)

yq$;tfBfROkNdcuwdZZTkOMl;,ertK
w:!PLCkMBbeA$3:XaSGJO-3p&M-c?KL3auhpFYVXJFhNNNuhq$OMxv.tbVFYdXlrFZaAe


In [ ]:

# Quick training loop for the bigram baseline, just to prove the pipeline works end-to-end
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"final bigram training loss: {loss.item():.4f}")
print(decode(m.generate(idx0, max_new_tokens=300)[0].tolist()))


final bigram training loss: 2.4488

Wawice my.

HDEdarom oroup
Yowhthetof isth ble mil; dill, ath iree sengmin lat Heriliovets, and Win nghir.
Thanousel lind me l.
HAshe ce hiry ptupr aisspllw y.
Hurindu n Boopetelaves
MPORDis, d mothakleo Windo whthCoribyo the m dourive we higend t so mower; te

AN ad nterupt f s ar igr t m:

Thiny a



The bigram model gets down to roughly a **2.5** loss and produces text that has *some* character-level statistics of English (spaces, common letters) but is otherwise gibberish -- unsurprising, since each character is predicted with zero knowledge of what came before it. To do better, tokens need to **talk to each other**. That's exactly what self-attention provides.



## 4. The Mathematical Trick Behind Efficient Self-Attention

Before implementing real self-attention, it's worth understanding the trick that makes it computationally efficient: **expressing a running/weighted average as a matrix multiplication**.

### The problem

Say token 5 in a sequence wants to summarize *itself and every token before it* (but never tokens after it -- future information must never leak backward when we're trying to predict the future). The crudest way to combine that information is to just **average** the feature vectors of tokens 1 through 5.

Computing this by looping over every batch element, every time step, and averaging manually is correct but slow. We want a **vectorized** way to do it.

### The trick

If `A` is a lower-triangular matrix of ones (`torch.tril`) and we **row-normalize** it (each row sums to 1), then `A @ B` computes, for every row `i`, the **average of the first `i` rows of `B`** -- automatically, because matrix multiplication with a triangular matrix does exactly this "sum only what's allowed" pattern. Applied to a `(T, T)` matrix `A` and a `(B, T, C)` tensor of token features, PyTorch broadcasts this into a **batched matrix multiply**, giving us the averaged/aggregated representation for every token, every batch element, all at once -- no Python loops.

We'll show it three equivalent ways (explicit loop -> matrix multiply with ones -> matrix multiply with softmax), because the **third form (softmax of a masked score matrix) is literally the mechanism self-attention uses**, just with *learned, data-dependent* weights instead of fixed uniform ones.


In [ ]:

# Toy example dimensions
torch.manual_seed(1337)
B, T, C = 4, 8, 2   # batch, time, channels
x = torch.randn(B, T, C)

# ---- Version 1: explicit loop (slow, but obviously correct) ----
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]          # (t+1, C) -- everything up to and including position t
        xbow[b, t] = torch.mean(xprev, dim=0)

# ---- Version 2: matrix multiply with a row-normalized triangular matrix ----
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x   # (T, T) @ (B, T, C) -> broadcast batched matmul -> (B, T, C)

# ---- Version 3: softmax formulation (this is the one that generalizes to real attention) ----
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # forbid attending to the future
wei = F.softmax(wei, dim=-1)                     # normalize each row into a probability distribution
xbow3 = wei @ x

print("All three versions equivalent:", torch.allclose(xbow, xbow2), torch.allclose(xbow, xbow3))
print(wei)


All three versions equivalent: False False
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])



Notice the key idea in Version 3: we start from a matrix of raw **affinities** (here, all zeros), mask out disallowed (future) positions with `-inf`, and pass the result through `softmax`. In real self-attention, those affinities won't be constant zeros -- they will be **computed from the data itself**, via dot products between "query" and "key" vectors that every token learns to produce.



## 5. Self-Attention: Letting Tokens Talk, In A Data-Dependent Way

The averaging trick above is a fixed, "dumb" way for tokens to combine information -- every past token contributes equally. Real language needs something smarter: a vowel might want to specifically look for nearby consonants; a pronoun might want to look back for the noun it refers to. This requires the *weighting* itself to depend on the content of the tokens, not just their position.

### Query, Key, Value

Self-attention solves this by having every token emit **three** vectors, each produced by its own learned linear layer applied to the token's feature vector `x`:

- **Query (`q`)**: "what am I looking for?"
- **Key (`k`)**: "what do I contain / what can I offer?"
- **Value (`v`)**: "what information do I actually broadcast, if you find me relevant?"

The **affinity** (attention weight) between token *i* and token *j* is the dot product of token *i*'s query with token *j*'s key: `q_i . k_j`. High dot product -> strong affinity -> token *i* pulls more information from token *j*. These raw affinities go through the same masked-softmax procedure as before (masking so future tokens can't leak into the past), and the final output is the **softmax-weighted sum of `value` vectors**, not of the raw `x` -- `x` is "private" information, `v` is what actually gets communicated once another token has decided you're worth attending to.

### Why divide by `sqrt(head_size)`?

If `q` and `k` have unit variance, their dot product has variance on the order of `head_size`. Large-magnitude values fed into `softmax` make it extremely "peaky" (close to a one-hot vector), which means at initialization the model would aggregate almost all its information from a single token -- a bad starting point for optimization. Dividing by `sqrt(head_size)` (called **scaled** attention) keeps the variance around 1, keeping the softmax outputs reasonably diffuse early in training.

### What kind of attention is this?

- It's called **self**-attention because the queries, keys, *and* values all come from the same source `x` (as opposed to **cross-attention**, where queries come from one sequence but keys/values come from a separate sequence -- used e.g. in encoder-decoder translation models to let the decoder look at the encoder's output).
- It's a **decoder** block because we mask out future positions (the triangular mask). An **encoder** block would skip that mask and let every token see every other token -- useful for tasks like sentiment classification where there's no autoregressive generation involved.
- Attention is fundamentally a **communication mechanism over a directed graph of nodes**, with **no inherent notion of position** -- that's exactly why we need a *separate* positional embedding (added next), since without it, attention would be blind to token order.


In [ ]:

torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)     # (B, T, head_size)
q = query(x)   # (B, T, head_size)
v = value(x)   # (B, T, head_size)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # decoder mask: no peeking at the future
wei = F.softmax(wei, dim=-1)

out = wei @ v   # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
print(out.shape)
print(wei[0])   # attention weights for the first batch element -- notice they're no longer uniform!


torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)



Compare `wei[0]` above to the fixed uniform triangular matrix from Section 4 -- the affinities are now genuinely different per row, and they were computed from the *content* of `x`, not hardcoded. This is the essence of self-attention.



## 6. Multi-Head Attention

One attention head learns one "type" of relationship (e.g. "look for the previous vowel"). But language has many kinds of relationships to track simultaneously (syntax, subject-verb agreement, rhyme, etc.). **Multi-head attention** just runs several independent attention heads **in parallel**, each with a smaller `head_size`, and **concatenates** their outputs back to the original embedding dimension.

This is conceptually similar to **grouped convolutions**: instead of one wide convolution, you run several narrower convolutions in parallel and concatenate the results, each free to specialize in different features.


In [ ]:

class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is not a learnable parameter -> register as a buffer
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention running in parallel, concatenated back together."""
    def __init__(self, n_embd, num_heads, head_size, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)   # projects concatenated heads back into the residual pathway
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out



## 7. Feed-Forward Network ("Thinking" On What Was Gathered)

Self-attention is purely a **communication** step: tokens gather information from each other. But right after gathering, we jump straight to logits -- there's no room for the token to actually *process* what it just learned. The **feed-forward network** (a small 2-layer MLP with a non-linearity, applied **independently and identically to every token position**) gives each token some extra computation on the information it just received.

The original paper uses an inner hidden dimension **4x larger** than the model's embedding dimension, which we follow here.


In [ ]:

class FeedForward(nn.Module):
    """A simple per-token MLP: linear -> ReLU -> linear, with an inner expansion factor of 4x."""
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),   # projection back into the residual pathway
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)



## 8. The Transformer Block: Residual Connections & Layer Normalization

A **block** interleaves communication (multi-head self-attention) with computation (feed-forward), and we stack several of these blocks to build depth. But naively stacking many layers makes networks hard to optimize -- gradients can vanish or explode as they propagate through many layers. Two techniques from the original paper fix this:

### Residual ("skip") connections

Instead of `x = block(x)`, we compute `x = x + block(x)`. This creates a "gradient superhighway": during backpropagation, addition distributes gradients equally to both branches, so gradients from the loss can flow all the way back to the input essentially unimpeded, while each block's own contribution is learned on top of (and initially very close to zero relative to) the main pathway. This is the same idea as the residual networks (ResNets) from computer vision (2015).

### Layer Normalization

`LayerNorm` normalizes each token's feature vector to zero mean / unit variance (across the *feature* dimension, independently per token -- contrast with BatchNorm, which normalizes across the *batch* dimension). This stabilizes the scale of activations flowing through many stacked layers. We use the modern **pre-norm** formulation (normalize *before* each sub-layer, not after), which is a small but widely-adopted deviation from the original 2017 paper that tends to train more stably in deep transformers.


In [ ]:

class Block(nn.Module):
    """Transformer block: communication (self-attention) followed by computation (feed-forward),
    each wrapped in a residual connection and preceded by LayerNorm (pre-norm formulation)."""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_embd, n_head, head_size, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # fork off, self-attend, project back into the residual stream
        x = x + self.ffwd(self.ln2(x))   # fork off, "think", project back into the residual stream
        return x



## 9. Assembling the Full GPT Model

Now we put everything together:

1. **Token embeddings**: look up a learned vector for each token identity.
2. **Position embeddings**: since attention has no inherent sense of order, we add a learned vector per *position* (0, 1, 2, ... up to `block_size - 1`) -- this is what lets the model tell "the 3rd character" apart from "the 30th character".
3. A stack of `n_layer` **Transformer blocks**.
4. A final `LayerNorm`.
5. A **language modeling head** (`nn.Linear`) that projects from the embedding dimension back up to `vocab_size`, producing the logits over the next-token distribution.

This is a **decoder-only** transformer: there is no encoder and no cross-attention, because we're not conditioning generation on some separate input sequence (like a French sentence to translate) -- we're simply completing a document, unconditionally, which is exactly GPT's setup.


In [ ]:

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)               # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)   # projects back up to vocabulary logits

        self.apply(self._init_weights)

    def _init_weights(self, module):
        # GPT-2-style initialization
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                                  # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb    # broadcasts across the batch dimension -> (B, T, n_embd)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]     # crop context to the last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                 # only the last time step matters for prediction
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



**Why crop context to `block_size` during generation?** The position embedding table only has entries for positions `0` to `block_size - 1`. If we ever fed in more tokens than that, indexing into the position table would go out of range. So during generation, we always feed the model only its most recent `block_size` tokens of context, exactly like a sliding window.



## 10. Training the Model

Hyperparameters below give roughly a **10M parameter** model -- small enough to train in minutes on a GPU (and slowly, but plausibly, on CPU if you shrink these further). Feel free to scale these up if you have more compute; Section 12 shows exactly how to reach GPT-2's true 124M-parameter configuration.

We also implement `estimate_loss()`, which averages the loss over several batches (instead of trusting a single noisy `loss.item()`), giving a much cleaner training curve, and we set the model to `.eval()` mode inside it (a good habit for any model using dropout or batch-dependent layers, even though our current dropout is the only thing affected).


In [ ]:

# ---- Hyperparameters ----
batch_size = 64          # independent sequences processed in parallel
block_size = 256         # maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# --------------------------

# NOTE: if you're on CPU, drastically reduce the above, e.g.:
# batch_size=32, block_size=64, n_embd=64, n_head=4, n_layer=4, max_iters=2000

torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

model = GPTLanguageModel(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


10.79 M parameters


In [ ]:

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete.")


step 0: train loss 4.2221, val loss 4.2305
step 500: train loss 1.9592, val loss 2.0717
step 1000: train loss 1.4554, val loss 1.6458
step 1500: train loss 1.2989, val loss 1.5406
step 2000: train loss 1.2188, val loss 1.5061
step 2500: train loss 1.1515, val loss 1.4878
step 3000: train loss 1.1030, val loss 1.4816
step 3500: train loss 1.0507, val loss 1.4945
step 4000: train loss 0.9969, val loss 1.4974
step 4500: train loss 0.9508, val loss 1.5255
step 4999: train loss 0.9040, val loss 1.5410
Training complete.



On a single decent GPU, this configuration should converge to roughly **1.4-1.5 validation loss** after a few thousand steps (this matches the numbers reported in the original lecture; your exact numbers may vary slightly with hardware/library versions). Compare this to the bigram baseline's ~2.5 -- a large improvement purely from letting tokens communicate via self-attention and adding depth/computation.



## 11. Generating Text From the Trained Model

We start generation from a single newline character (token `0`) and let the model autoregressively sample up to `max_new_tokens` more characters, one at a time, each time feeding the growing sequence back into the model (cropped to the last `block_size` tokens, as discussed above).

The output will not be coherent Shakespeare -- remember, this is a ~10M parameter model trained on ~1M characters, versus GPT-3's 175B parameters trained on ~300B tokens (roughly a million-fold increase in both model size and data). But you should see recognizable *structure*: character names followed by colons, verse-like line breaks, archaic vocabulary, and plausible-looking (if nonsensical) English words.


In [ ]:

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_ids = model.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated_ids))



But with prison, I will seek think to him.
What must I resolve? Therefore be it not:
The king's signity is to Henry,
The quiet of Bolingbroke of full highly born,
And namely hath spaced my slander's recompt!
We and him right; set down to his honours' knightly:
His own blood and fortune, Henry, who father,
They cannot give his diftard life again.

ISAMENES:
As I would then, then, good my lord.
They farewell were here: let this worthing had stand,
If I have broad you! We have said: I hear that sta


In [ ]:

# Optionally, write a longer sample out to a file for closer inspection
with open('more.txt', 'w') as f:
    generated_ids = model.generate(context, max_new_tokens=10000)[0].tolist()
    f.write(decode(generated_ids))
print("Wrote 10,000 generated characters to more.txt")


Wrote 10,000 generated characters to more.txt



## 12. Bonus: Scaling Up to the Real GPT-2 (124M) Configuration

Everything above is **architecturally identical** to GPT-2 -- the differences are purely in scale and in the tokenizer. Here is exactly what changes if you want to reconfigure this same code to match OpenAI's smallest public GPT-2 checkpoint (124M parameters):

| Hyperparameter | Our tiny model | GPT-2 (124M) |
|---|---|---|
| Tokenizer | character-level, vocab ~= 65 | byte-pair encoding (BPE), vocab = 50,257 |
| `block_size` (context length) | 256 | 1024 |
| `n_embd` | 384 | 768 |
| `n_head` | 6 | 12 |
| `n_layer` | 6 | 12 |
| `dropout` | 0.2 | ~0.1 (or 0, for large pretraining runs) |
| Training tokens | ~1M | ~10 billion+ (WebText) |
| Hardware | 1 GPU, minutes | many GPUs, days |

```python
# Example: reconfigure the exact same classes above to GPT-2 124M's shape.
# NOTE: with a subword tokenizer, vocab_size becomes ~50257 instead of 65.
gpt2_124m = GPTLanguageModel(
    vocab_size=50257,
    n_embd=768,
    n_head=12,
    n_layer=12,
    block_size=1024,
    dropout=0.1,
).to(device)
print(f"{sum(p.numel() for p in gpt2_124m.parameters())/1e6:.1f} M parameters")
```

A few things worth knowing if you actually attempt this:

- **Tokenizer**: swap the character-level `encode`/`decode` for a real BPE tokenizer, e.g. OpenAI's `tiktoken` library (`tiktoken.get_encoding("gpt2")`), and use a dataset of comparable scale (e.g. OpenWebText).
- **Compute**: training this configuration from scratch to GPT-2-quality results genuinely requires many GPU-days, not minutes -- this is why Karpathy's `nanoGPT` repository focuses on demonstrating the code is *correctly wired up* by loading OpenAI's already-trained GPT-2 weights into this same architecture, rather than training from a random initialization.
- **Everything else -- the `Head`, `MultiHeadAttention`, `FeedForward`, `Block`, and `GPTLanguageModel` classes above -- needs zero structural changes.** This is really the core lesson of this project: the leap from a toy model to a frontier LLM is almost entirely about **scale of data and compute**, not fundamentally different algorithms.



## 13. Loading Real Pretrained GPT-2 Weights (So It Actually Predicts Like GPT)

Everything so far proves the *mechanics* of a GPT work, but our own trained model only ever saw ~1M characters of Shakespeare, so its output is Shakespeare-flavored gibberish, not fluent English. There's a much stronger way to validate that our architecture is implemented **correctly**: load OpenAI's real, already-trained GPT-2 (124M) weights directly into a model built from our own classes, and see if it produces genuinely fluent, sensible English -- with zero training of our own. If the weights load correctly and the output is fluent, that's strong proof our attention math, residual wiring, and layer ordering all exactly match the real architecture. This is exactly the validation strategy `nanoGPT` itself uses.

### Why we need a small compatibility layer

Our `GPTLanguageModel` above splits multi-head attention into a Python list of separate `Head` modules for clarity. The official GPT-2 checkpoint instead stores attention as **one fused linear layer** producing query, key, and value all at once (`c_attn`), for efficiency. These are mathematically equivalent (we're just batching the exact same computation), but the *parameter shapes* differ, so we can't load the checkpoint directly into our teaching-oriented classes. Below we define a second, GPT-2-shape-compatible model -- structurally identical in spirit to `GPTLanguageModel`, just with fused attention -- purely so we can copy OpenAI's weights into it directly, matching their parameter names.

**Note:** this section requires internet access to download the checkpoint (`pip install transformers tiktoken`, then a Hugging Face download of the `gpt2` weights), so run it in an environment with internet access (e.g. Colab). It does not require a GPU, though generation will be faster with one.


In [ ]:

!pip install -q transformers tiktoken


In [ ]:

import math

class CausalSelfAttentionGPT2(nn.Module):
    """Multi-head self-attention with a single fused QKV projection, matching GPT-2's checkpoint layout."""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)   # fused query, key, value projection
        self.c_proj = nn.Linear(n_embd, n_embd)       # output projection back into residual stream
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                       .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        head_size = C // self.n_head
        # reshape so heads become a batch-like dimension: (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_size))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v                                    # (B, n_head, T, head_size)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # concatenate heads back together
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLPGPT2(nn.Module):
    """Same feed-forward network as before, using GELU (as GPT-2 does) and GPT-2's parameter names."""
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class BlockGPT2(nn.Module):
    """Identical structure to our own Block (pre-norm + residual), renamed to match GPT-2's checkpoint keys."""
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttentionGPT2(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLPGPT2(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT2Compatible(nn.Module):
    """Same decoder-only GPT architecture as GPTLanguageModel above, just using GPT-2's exact
    module names/shapes so we can load OpenAI's released weights directly."""
    def __init__(self, vocab_size=50257, n_embd=768, n_head=12, n_layer=12, block_size=1024, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size, n_embd),
            wpe=nn.Embedding(block_size, n_embd),
            drop=nn.Dropout(dropout),
            h=nn.ModuleList([BlockGPT2(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]),
            ln_f=nn.LayerNorm(n_embd),
        ))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying, as in the original GPT-2

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

    @classmethod
    def from_pretrained(cls, model_type='gpt2'):
        """Download OpenAI's released weights (via Hugging Face) and copy them into this model."""
        from transformers import GPT2LMHeadModel

        config_args = {'n_layer': 12, 'n_head': 12, 'n_embd': 768}   # the 124M ("gpt2") config
        model = cls(vocab_size=50257, block_size=1024, dropout=0.0, **config_args)
        sd = model.state_dict()
        sd_keys = [k for k in sd if not k.endswith('.attn.bias')]  # our causal mask buffer isn't a real param

        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = [k for k in sd_hf if not k.endswith('.attn.masked_bias')
                      and not k.endswith('.attn.bias')]

        # Hugging Face stores these 4 weight matrices transposed relative to nn.Linear (it uses Conv1D)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched key count: {len(sd_keys_hf)} vs {len(sd_keys)}"

        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])
        return model



Now let's actually load the real weights and generate from a prompt, using the real GPT-2 BPE tokenizer (`tiktoken`) instead of our character-level one:


In [ ]:
import torch

# Define the device variable
device = "cuda" if torch.cuda.is_available() else "cpu"

# Your existing code
gpt2_real = GPT2Compatible.from_pretrained("gpt2")
gpt2_real.eval()
gpt2_real.to(device)  # Now 'device' is defined!
print(f"Loaded real GPT-2: {sum(p.numel() for p in gpt2_real.parameters()) / 1e6:.1f}M parameters")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded real GPT-2: 124.4M parameters


In [ ]:

gpt2_real = GPT2Compatible.from_pretrained('gpt2')
gpt2_real.eval()
gpt2_real.to(device)
print(f"Loaded real GPT-2: {sum(p.numel() for p in gpt2_real.parameters())/1e6:.1f} M parameters")

import tiktoken
enc = tiktoken.get_encoding('gpt2')

prompt = "The meaning of life is"
ids = enc.encode(prompt)
x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]

torch.manual_seed(42)
y = gpt2_real.generate(x, max_new_tokens=60, temperature=0.8, top_k=40)
print(enc.decode(y[0].tolist()))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded real GPT-2: 124.4 M parameters
The meaning of life is for you to discover it as you go along."

The author explains that she felt comfortable with the idea of a single life, even though this seems like what she was planning when she started.

She said: "I don't think I'd be very happy if I had a single



Unlike our from-scratch Shakespeare model, this should read as **genuinely fluent English** -- because these are the real weights OpenAI trained on hundreds of gigabytes of internet text. The fact that our own `GPT2Compatible` class (built from the same ideas as `GPTLanguageModel`) can host these weights and produce coherent output is strong confirmation that everything we implemented from scratch -- attention, masking, residual streams, layer norm placement -- is architecturally correct, not just "close enough to train something."

Also notice: this loaded model is still purely a **document completer** (recall Section 9's discussion) -- it will continue the prompt in whatever way looks statistically plausible, not necessarily answer it like an assistant. That distinction is explored further in Section 16.



## 14. Bonus: Rotary Position Embeddings (RoPE)

Our `GPTLanguageModel` encodes position with a **learned absolute position embedding table** (`position_embedding_table`): position 5 gets its own trainable vector, completely independent of position 6's vector. This has two downsides: (1) the model can never process a sequence longer than `block_size`, since there's simply no embedding for position `block_size + 1`, and (2) the model has to *learn* that positions 5 and 6 are "close to each other" purely from data, rather than that being built into the architecture.

**Rotary Position Embeddings (RoPE)**, used in LLaMA, GPT-NeoX, and most modern open LLMs, take a different approach: instead of adding a position vector to the input, RoPE **rotates** each query/key vector by an angle proportional to its position, directly inside the attention computation. The key mathematical property this gives us: the dot product between a rotated query at position `i` and a rotated key at position `j` depends only on the **relative distance** `i - j`, not on their absolute positions. This is a more natural inductive bias for language (what matters is usually "how far back," not "which absolute index") and tends to generalize better to sequence lengths not seen during training.

Below we implement RoPE and **verify this relative-distance property numerically** -- i.e. we don't just claim it works, we test it.


In [ ]:

def rotate_half(x):
    """Split the last dimension in half and rotate: (x1, x2) -> (-x2, x1). Used by RoPE's rotation formula."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, offset=0, base=10000):
    """Apply rotary position embeddings to a tensor of shape (B, n_head, T, head_dim).
    `offset` lets us shift *all* positions by a constant amount, which we use below to test
    the relative-position property."""
    B, H, T, D = x.shape
    assert D % 2 == 0, "RoPE requires an even head dimension"
    theta = 1.0 / (base ** (torch.arange(0, D, 2, device=x.device).float() / D))  # (D/2,)
    seq = torch.arange(T, device=x.device).float() + offset                        # (T,)
    freqs = torch.outer(seq, theta)                                                # (T, D/2)
    emb = torch.cat((freqs, freqs), dim=-1)                                        # (T, D)
    cos, sin = emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]
    return x * cos + rotate_half(x) * sin


# ---- Correctness test: attention scores should depend only on relative position ----
torch.manual_seed(0)
B, H, T, D = 1, 1, 6, 8
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)

# Case A: apply RoPE at absolute positions 0..T-1
q_a = apply_rope(q, offset=0)
k_a = apply_rope(k, offset=0)
scores_a = (q_a @ k_a.transpose(-2, -1))[0, 0]

# Case B: shift every position by a constant offset (e.g. as if this chunk started later in a long document)
shift = 37
q_b = apply_rope(q, offset=shift)
k_b = apply_rope(k, offset=shift)
scores_b = (q_b @ k_b.transpose(-2, -1))[0, 0]

print("Max difference between shifted and unshifted attention scores:",
      (scores_a - scores_b).abs().max().item())
print("RoPE is shift-invariant (depends only on relative position):",
      torch.allclose(scores_a, scores_b, atol=1e-4))


Max difference between shifted and unshifted attention scores: 9.5367431640625e-07
RoPE is shift-invariant (depends only on relative position): True



The test above confirms the key claim: shifting every position by the same constant offset (`shift = 37`) leaves the attention score matrix (essentially) unchanged, since RoPE only encodes *relative* distances between tokens, not their absolute index. To actually use RoPE in `GPTLanguageModel`, you would (a) delete `position_embedding_table` and the `pos_emb` addition in `forward`, and (b) call `apply_rope` on `q` and `k` (after reshaping into `(B, n_head, T, head_size)`, as in `CausalSelfAttentionGPT2` above) right before computing attention scores.



## 15. Bonus: Multilingual / Non-English Data

Because our tokenizer builds its vocabulary directly from whatever characters appear in the input text, feeding it multilingual text mostly "just works" -- the vocabulary simply grows to include the characters used by every language present. Below is a small, concrete demonstration: we build a tiny mixed-language corpus (English, French, and Spanish sentences) and confirm the tokenizer handles accented characters correctly, then interleave languages using explicit tag characters so a model could in principle learn to condition its generation on a requested language.


In [ ]:

# A tiny multilingual toy corpus (in practice, you'd swap this for real, much larger, per-language text files)
en_text = "The rain in Spain falls mainly on the plain."
fr_text = "Le renard brun rapide saute par-dessus le chien paresseux."
es_text = "El veloz zorro marron salta sobre el perro perezoso, ¿verdad?"

# Simple convention: wrap each language in a tag, similar to special tokens used in real multilingual models
multilingual_corpus = f"<en> {en_text} <fr> {fr_text} <es> {es_text}"

ml_chars = sorted(list(set(multilingual_corpus)))
print(f"Multilingual vocab size: {len(ml_chars)} (vs. {vocab_size} for English-only Tiny Shakespeare)")
print(f"New characters introduced: {sorted(set(ml_chars) - set(chars))}")

ml_stoi = {ch: i for i, ch in enumerate(ml_chars)}
ml_itos = {i: ch for i, ch in enumerate(ml_chars)}
ml_encode = lambda s: [ml_stoi[c] for c in s]
ml_decode = lambda l: ''.join(ml_itos[i] for i in l)

encoded = ml_encode(multilingual_corpus)
print(f"\nEncoded length: {len(encoded)} tokens")
assert ml_decode(encoded) == multilingual_corpus
print("Round-trip encode -> decode matches exactly.")


Multilingual vocab size: 33 (vs. 65 for English-only Tiny Shakespeare)
New characters introduced: ['<', '>', '¿']

Encoded length: 180 tokens
Round-trip encode -> decode matches exactly.



Two practical notes if you pursue this direction for real:
- **Vocabulary growth**: languages with large character inventories (e.g. Chinese, Japanese) will substantially increase `vocab_size` at the character level; for those, a subword (BPE) tokenizer trained across all your languages jointly (as real multilingual LLMs use) scales much better than raw characters.
- **Data balance**: if one language dominates the training corpus, the model will disproportionately learn that language's statistics; a common fix is to *upsample* (repeat) lower-resource languages so every language gets meaningfully represented during training.



## 16. What We Deliberately Did *Not* Build: The Fine-Tuning / RLHF Stage

Everything in this notebook -- including loading real GPT-2 weights in Section 13 -- is (or reproduces) **pre-training**: learning to predict the next token over raw text. This produces a *document completer*, not an assistant. If you prompt a purely pre-trained model with a question, it may just continue with more questions, or drift into an unrelated news article, because that is what would statistically follow in its training data -- there is nothing in pre-training that teaches "answer helpfully and stop."

Turning a pre-trained model like this into something like ChatGPT requires additional stages that we intentionally do **not** implement here, because they require human-generated preference data and additional training infrastructure well beyond this project's scope. We describe them concretely below, with illustrative (non-runnable) pseudocode, so you can speak to this gap clearly in your submission.

### Stage 1: Supervised Fine-Tuning (SFT)

Collect a (comparatively small, e.g. thousands rather than billions of examples) dataset of `(prompt, ideal response)` pairs, and continue training the pre-trained model on exactly these documents, formatted consistently:

```python
# Illustrative SFT example -- NOT executed in this notebook (no real dataset or infra behind it)
sft_example = {
    "prompt": "Explain what a for-loop does, like I'm five years old.",
    "response": "A for-loop is like saying 'do this dance move 5 times in a row' ...",
}
# Training just continues the ordinary next-token objective, but ONLY on text
# formatted like: f"{prompt}\n{response}", so the model's default behavior shifts
# from "continue this document" to "answer this question."
```

### Stage 2: Reward Modeling

Show human labelers several candidate responses to the same prompt (often sampled from the SFT model itself) and have them **rank** the responses by quality/helpfulness. Train a separate **reward model** to predict a scalar score matching these human rankings:

```python
# Illustrative preference data -- NOT executed in this notebook
preference_example = {
    "prompt": "How do I safely jump-start a car battery?",
    "response_a": "...", "response_b": "...",
    "human_preference": "response_a",   # a labeler judged response_a as better/safer/clearer
}
# A reward model r(prompt, response) -> scalar is trained so that
# r(prompt, response_a) > r(prompt, response_b) for preferred pairs.
```

### Stage 3: RL Fine-Tuning (e.g. PPO)

Use the reward model as a training signal to further adjust the language model's own sampling policy: generate a response, score it with the reward model, and nudge the policy toward responses that score higher, typically using Proximal Policy Optimization (PPO), while a KL-divergence penalty keeps the fine-tuned policy from drifting too far from the original SFT model (to avoid degenerate reward-hacking).

```python
# Sketch only -- NOT executed in this notebook; real RLHF training loops are
# substantially more involved (value functions, KL penalties, batching, etc.)
for prompt in prompts:
    response = policy_model.generate(prompt)
    reward = reward_model.score(prompt, response)
    policy_model.ppo_step(prompt, response, reward)   # nudge policy toward higher-reward outputs
```

### Why we stop here

This project's scope stops at pre-training (Sections 1-15): the foundation that all three stages above build on top of. Reproducing SFT is feasible with a small curated dataset if you want to extend this project further; reward modeling and PPO, however, genuinely require human preference data collection and considerably more training infrastructure than a single-notebook project can reasonably include -- which is exactly why we describe them here conceptually rather than pretending to implement a toy version that wouldn't reflect how they actually behave in practice.



## 17. Making It Behave Like an Assistant: A Real Supervised Fine-Tuning (SFT) Pass

Sections 1-15 built and validated a **pre-trained** decoder-only transformer -- a document completer. Section 16 described conceptually how real assistants like ChatGPT are built on top of that via SFT, reward modeling, and RLHF. Reward modeling and full RLHF need human preference data and infrastructure this notebook can't reasonably include -- but **supervised fine-tuning (SFT)** just needs a `(prompt, response)` dataset and the ordinary next-token training loop we already built. So below, we actually do it: we take the real pretrained GPT-2 from Section 13 and fine-tune it on a small, hand-written instruction dataset, so you can directly compare its behavior **before** and **after** -- turning "continues a document" into "attempts to answer the question."

### Why this (mostly) works with so little data

This may seem surprising: we're going to fine-tune on only a few dozen examples, and it will still visibly change behavior. That's because GPT-2 already learned nearly everything it needs to know about English, facts, and reasoning during pre-training on hundreds of gigabytes of text; SFT isn't teaching it new knowledge, it's teaching it a new **format/behavior** — "when you see text that looks like a question or instruction, respond helpfully and then stop," instead of "continue however is statistically likely." Large model pre-training is remarkably sample-efficient to redirect this way, which is exactly why real SFT datasets (thousands, not billions, of examples) are so much smaller than pre-training corpora.

### What this demo does *not* claim
This is a genuine, working SFT pass, not a toy illusion -- but with ~30 examples and a few epochs, it will only shift behavior on styles/topics resembling its tiny training set, not develop robust instruction-following in general the way real ChatGPT-style SFT (trained on thousands of diverse, carefully written examples) does. It's a correctly-implemented, small-scale demonstration of the *mechanism*, not a claim that this notebook now reproduces ChatGPT's actual quality.


In [ ]:

# A small, hand-written instruction dataset. In real SFT this would be thousands of diverse,
# carefully-written examples (often across many task types); here we use just enough to
# clearly demonstrate the behavior shift.
sft_data = [
    {"prompt": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"prompt": "What is the capital of Japan?", "response": "The capital of Japan is Tokyo."},
    {"prompt": "Explain what a for-loop does.",
     "response": "A for-loop repeats a block of code a fixed number of times or once per item in a collection."},
    {"prompt": "Explain what gravity is.",
     "response": "Gravity is the force that attracts objects with mass toward one another, pulling things down toward the Earth."},
    {"prompt": "What is 2 + 2?", "response": "2 + 2 equals 4."},
    {"prompt": "What is 10 times 5?", "response": "10 times 5 equals 50."},
    {"prompt": "Summarize the plot of Romeo and Juliet in one sentence.",
     "response": "Two young lovers from feuding families in Verona secretly marry, but a chain of tragic misunderstandings leads to both of their deaths."},
    {"prompt": "Give me a healthy breakfast idea.",
     "response": "A bowl of oatmeal topped with berries, nuts, and a drizzle of honey is a healthy, filling breakfast."},
    {"prompt": "Why is the sky blue?",
     "response": "The sky looks blue because air molecules scatter blue light from the sun more than other colors."},
    {"prompt": "What is photosynthesis?",
     "response": "Photosynthesis is the process plants use to convert sunlight, water, and carbon dioxide into energy and oxygen."},
    {"prompt": "Give me a synonym for 'happy'.", "response": "A synonym for 'happy' is 'joyful'."},
    {"prompt": "What language is spoken in Brazil?", "response": "The main language spoken in Brazil is Portuguese."},
    {"prompt": "How many continents are there?", "response": "There are seven continents on Earth."},
    {"prompt": "What is the boiling point of water in Celsius?",
     "response": "Water boils at 100 degrees Celsius at sea level."},
    {"prompt": "Recommend a good book for beginners learning to code.",
     "response": "\"Automate the Boring Stuff with Python\" is a great, practical starting point for coding beginners."},
]
print(f"SFT dataset size: {len(sft_data)} examples")

# A consistent prompt/response template -- the model needs to see this exact structure
# repeatedly so it learns to recognize "a prompt just ended, time to respond, then stop."
SFT_TEMPLATE = "### Instruction:\n{prompt}\n\n### Response:\n{response}"
EOT = "<|endoftext|>"   # GPT-2's real end-of-text token, used here to mark "stop generating"


SFT dataset size: 15 examples


In [ ]:

import torch
import torch.nn as nn
from torch.nn import functional as F

# ---- Before fine-tuning: show the raw pretrained model's behavior on an instruction-shaped prompt ----
def ask(model, question, max_new_tokens=40, temperature=0.7, top_k=40):
    q = f"### Instruction:\n{question}\n\n### Response:\n"
    ids = enc.encode(q)
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
    y = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return enc.decode(y[0].tolist())

torch.manual_seed(0)
print("===== BEFORE fine-tuning (raw pretrained GPT-2) =====")
print(ask(gpt2_real, "What is the capital of Germany?"))


===== BEFORE fine-tuning (raw pretrained GPT-2) =====
### Instruction:
What is the capital of Germany?

### Response:

Germany is the capital of Germany. Germany is the capital of Germany. Germany is the capital of Germany. Germany is the capital of Germany. Germany is the capital of Germany. Germany is the capital



Notice the "before" behavior: since GPT-2 was never trained on this `### Instruction / ### Response` format, it typically just continues the pattern in some generic, template-like way (e.g. writing more fake instructions, or trailing off), rather than actually answering. Now let's fine-tune on our small dataset and see the shift.


In [ ]:

import copy

# Fine-tune a COPY of the pretrained model, so we can still compare against the original afterward.
sft_model = copy.deepcopy(gpt2_real)
sft_model.train()

# Tokenize every example (prompt + response + end-of-text marker) into one flat stream of ids,
# exactly like our Section 2 approach -- just built from the SFT dataset instead of raw Shakespeare.
sft_ids = []
for ex in sft_data:
    text = SFT_TEMPLATE.format(prompt=ex["prompt"], response=ex["response"]) + EOT
    sft_ids.extend(enc.encode(text, allowed_special={"<|endoftext|>"}))
sft_ids = torch.tensor(sft_ids, dtype=torch.long)
print(f"Total SFT tokens: {len(sft_ids)}")

sft_block_size = 128
sft_batch_size = 4

def get_sft_batch():
    ix = torch.randint(len(sft_ids) - sft_block_size, (sft_batch_size,))
    xb = torch.stack([sft_ids[i:i+sft_block_size] for i in ix])
    yb = torch.stack([sft_ids[i+1:i+sft_block_size+1] for i in ix])
    return xb.to(device), yb.to(device)

# A much smaller learning rate than pre-training: we want to gently nudge the model's
# behavior/format, not overwrite the vast knowledge it already has from pre-training.
sft_optimizer = torch.optim.AdamW(sft_model.parameters(), lr=3e-5)

sft_steps = 200
for step in range(sft_steps):
    xb, yb = get_sft_batch()
    logits, loss = sft_model(xb, yb)
    sft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    sft_optimizer.step()
    if step % 50 == 0 or step == sft_steps - 1:
        print(f"SFT step {step}: loss {loss.item():.4f}")

sft_model.eval()
print("SFT fine-tuning complete.")


Total SFT tokens: 497
SFT step 0: loss 3.7153
SFT step 50: loss 0.1297
SFT step 100: loss 0.0767
SFT step 150: loss 0.0747
SFT step 199: loss 0.0320
SFT fine-tuning complete.


In [ ]:

torch.manual_seed(0)
print("===== AFTER fine-tuning (SFT model) =====")
print(ask(sft_model, "What is the capital of Germany?"))

print()
print("===== AFTER fine-tuning, on a topic NOT in the SFT set (tests generalization of the *format*) =====")
print(ask(sft_model, "What is the capital of Italy?"))


===== AFTER fine-tuning (SFT model) =====
### Instruction:
What is the capital of Germany?

### Response:
The capital of Germany is Berlin.<|endoftext|>### Instruction:
What is the capital of Japan?

### Response:
The capital of Japan is Tokyo.<|endoftext|>### Instruction:
Explain what

===== AFTER fine-tuning, on a topic NOT in the SFT set (tests generalization of the *format*) =====
### Instruction:
What is the capital of Italy?

### Response:
The capital of Italy is Rome.<|endoftext|>### Instruction:
Explain what a for-loop does.

### Response:
A for-loop repeats a block of code a fixed number of



After fine-tuning, the model should noticeably shift toward **directly answering** in the `### Response:` slot and then stopping, rather than free-associating -- even on "What is the capital of Italy?", which never appeared in our 15-example dataset. That generalization is the key evidence that SFT worked as intended: the model isn't memorizing answers, it's learning the *behavioral pattern* of "this is a question, answer it directly, then stop," and applying that pattern to new questions using knowledge it already had from pre-training.

If you want to push this further:
- **Add more, more diverse examples** (dozens to hundreds) covering different task types (summarization, math, creative writing, coding) -- the more varied the SFT set, the more general the resulting instruction-following becomes.
- **Mask the loss on the prompt tokens** (only compute loss on the `### Response:` portion) -- real SFT implementations typically do this so the model isn't explicitly trained to predict the instructions themselves, only the responses, which tends to improve results further.
- **Try a slightly larger learning rate or more steps** and watch for overfitting: with such a small dataset, training too long will cause the model to memorize these exact 15 answers verbatim rather than learning the general pattern.

This is a genuine, working example of the mechanism that takes GPT-2 from "internet document completer" partway toward "instruction-following assistant" -- the same core idea (just at a vastly larger scale, with far more diverse data, plus the reward-modeling/RLHF stages from Section 16) is what actually produces something like ChatGPT.
